# 60M Parameter Transformer Model Training
## Google Colab Edition

This notebook trains a 60M parameter transformer model from scratch using only Python and NumPy.

**Features:**
- Production-ready transformer architecture
- Checkpoint saving and loading
- Free HuggingFace datasets
- GPU acceleration support

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q numpy datasets transformers torch

## 2. Clone Repository

In [ ]:
import os
from pathlib import Path

# Clone the repository
!git clone https://github.com/joshkenya/Llm_kenya.git
%cd Llm_kenya

print("Repository cloned successfully!")
print(f"Current directory: {os.getcwd()}")
print(f"Files: {os.listdir()}")

## 3. Import Model Components

In [ ]:
import sys
sys.path.append('/content/Llm_kenya')

from model import TransformerModel, Config, Tokenizer, Trainer
import numpy as np
from datasets import load_dataset

print("Model components imported successfully!")

## 4. Initialize Model

Initialize a 60M parameter transformer model with default configuration.

In [ ]:
# Model configuration for 60M parameters
config = Config(
    vocab_size=50257,
    max_position_embeddings=2048,
    hidden_size=768,
    num_hidden_layers=12,
    num_attention_heads=12,
    intermediate_size=3072,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

# Initialize model
model = TransformerModel(config)

print(f"Model initialized!")
print(f"Total Parameters: {model.total_params:,}")
print(f"Total Parameters (Millions): {model.total_params / 1e6:.2f}M")
print(f"\nModel Configuration:")
print(f"  Hidden Size: {config.hidden_size}")
print(f"  Number of Layers: {config.num_hidden_layers}")
print(f"  Number of Attention Heads: {config.num_attention_heads}")
print(f"  Vocabulary Size: {config.vocab_size}")
print(f"  Max Sequence Length: {config.max_position_embeddings}")

## 5. Load Dataset from HuggingFace

Using the 'wikitext' dataset (free and open source)

In [ ]:
# Load dataset from HuggingFace
print("Loading dataset from HuggingFace...")
dataset = load_dataset('wikitext', 'wikitext-2')

print(f"Dataset loaded!")
print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")

# Display sample
print(f"\nSample text:")
print(dataset['train'][0]['text'][:200])

## 6. Prepare Training Data

In [ ]:
# Initialize tokenizer
tokenizer = Tokenizer(config.vocab_size)

def prepare_batch(texts, tokenizer, max_length=2048, batch_size=2):
    """
    Prepare batch of tokenized data
    """
    input_ids = []
    target_ids = []
    
    for text in texts:
        # Tokenize
        tokens = tokenizer.encode(text)
        
        # Split into sequences of max_length
        for i in range(0, len(tokens) - max_length, max_length):
            seq = tokens[i:i + max_length]
            
            # Pad if necessary
            if len(seq) < max_length:
                seq = seq + [tokenizer.pad_token_id] * (max_length - len(seq))
            
            input_ids.append(seq[:-1])  # Input: all but last token
            target_ids.append(seq[1:])  # Target: all but first token
            
            if len(input_ids) >= batch_size:
                yield (
                    np.array(input_ids, dtype=np.int32),
                    np.array(target_ids, dtype=np.int32)
                )
                input_ids = []
                target_ids = []

print("Data preparation function created.")

## 7. Training Loop

In [ ]:
# Training configuration
num_epochs = 1  # Start with 1 epoch for demo
batch_size = 2
learning_rate = 1e-4
checkpoint_dir = './checkpoints'
checkpoint_interval = 100  # Save checkpoint every N steps

# Create trainer
trainer = Trainer(model, learning_rate=learning_rate)

# Create checkpoint directory
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

# Prepare training data
train_texts = [text for text in dataset['train']['text'] if len(text) > 100]
train_texts = train_texts[:1000]  # Use first 1000 samples for training

print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Batch Size: {batch_size}")
print(f"  Learning Rate: {learning_rate}")
print(f"  Training Samples: {len(train_texts)}")
print(f"  Checkpoint Directory: {checkpoint_dir}")

In [ ]:
# Training loop
print("\n" + "="*60)
print("Starting training...")
print("="*60)

global_step = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    epoch_loss = 0.0
    step_in_epoch = 0
    
    for batch_input_ids, batch_target_ids in prepare_batch(train_texts, tokenizer, max_length=256, batch_size=batch_size):
        try:
            # Training step
            loss = trainer.train_step(batch_input_ids, batch_target_ids)
            epoch_loss += loss
            step_in_epoch += 1
            global_step += 1
            
            # Print progress
            if global_step % 10 == 0:
                avg_loss = epoch_loss / step_in_epoch
                print(f"  Step {global_step} | Loss: {loss:.4f} | Avg Loss: {avg_loss:.4f}")
            
            # Save checkpoint
            if global_step % checkpoint_interval == 0:
                checkpoint_path = f"{checkpoint_dir}/checkpoint_step_{global_step}"
                model.save_checkpoint(checkpoint_path)
                print(f"  Checkpoint saved: {checkpoint_path}")
        
        except Exception as e:
            print(f"Error in training step: {e}")
            break
    
    avg_epoch_loss = epoch_loss / max(step_in_epoch, 1)
    print(f"\nEpoch {epoch + 1} completed. Average Loss: {avg_epoch_loss:.4f}")

print("\n" + "="*60)
print("Training completed!")
print("="*60)

## 8. Save Final Model

In [ ]:
# Save final model
final_checkpoint_path = './checkpoints/final_model'
model.save_checkpoint(final_checkpoint_path)
print(f"Final model saved to: {final_checkpoint_path}")

# Download checkpoint to local machine
import shutil
shutil.make_archive('llm_kenya_checkpoint', 'zip', './checkpoints')
print("Checkpoint archived as 'llm_kenya_checkpoint.zip'")

## 9. Training Results

In [ ]:
# Display training results
import matplotlib.pyplot as plt

print("Training Results:")
print(f"Total Steps: {len(trainer.losses)}")
print(f"Initial Loss: {trainer.losses[0]:.4f}" if trainer.losses else "N/A")
print(f"Final Loss: {trainer.losses[-1]:.4f}" if trainer.losses else "N/A")
print(f"Model Info: {trainer.get_model_info()}")

# Plot training loss
if len(trainer.losses) > 1:
    plt.figure(figsize=(10, 6))
    plt.plot(trainer.losses)
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('Training Loss Over Time')
    plt.grid(True)
    plt.show()

## 10. Test Generation

In [ ]:
# Test model generation
from chat import ChatBot

# Initialize chatbot with trained model
chatbot = ChatBot(model_path=final_checkpoint_path, config=config)

print("\nModel loaded for inference!")
print("Model Info:")
info = chatbot.get_model_info()
for key, value in info.items():
    print(f"  {key}: {value}")

In [ ]:
# Generate some test responses
test_prompts = [
    "Hello, how are you?",
    "What is machine learning?",
    "Tell me about transformers."
]

print("\nTest Generation:")
print("="*60)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    response = chatbot.chat(prompt)
    print(f"Response: {response}")
    print("-"*60)

## 11. Push to GitHub (Optional)

Uncomment and run to push trained model checkpoint to GitHub

In [ ]:
# Configure git and push checkpoint
# Uncomment to use

# !git config --global user.email "your-email@example.com"
# !git config --global user.name "Your Name"
# !git add checkpoints/
# !git commit -m "Add trained model checkpoint"
# !git push

print("To push to GitHub:")
print("1. Uncomment the code above")
print("2. Replace email and name with your GitHub credentials")
print("3. Run the cell")